# Machine Learning Models

This notebook estimates machine-learning models for UK quarterly GDP nowcasting using the same expanding-window pseudo-real-time framework as the benchmark models.

Random Forest and Gradient Boosting models are evaluated separately across the M1, M2 and M3 information sets. At each forecast quarter, models are estimated using only observations available before that quarter.

In [1]:
import pandas as pd

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

In [2]:
dataset_m1 = pd.read_csv("../data/processed/dataset_m1.csv")
dataset_m2 = pd.read_csv("../data/processed/dataset_m2.csv")
dataset_m3 = pd.read_csv("../data/processed/dataset_m3.csv")

for df in [dataset_m1, dataset_m2, dataset_m3]:
    df["quarter"] = pd.PeriodIndex(df["quarter"], freq="Q")

In [3]:
print("M1:", dataset_m1.shape)
print("M2:", dataset_m2.shape)
print("M3:", dataset_m3.shape)

print(
    "Sample:",
    dataset_m1["quarter"].min(),
    "to",
    dataset_m1["quarter"].max()
)

M1: (81, 4)
M2: (81, 8)
M3: (81, 20)
Sample: 2006Q1 to 2026Q1


In [4]:
evaluation_start = pd.Period("2017Q1", freq="Q")

evaluation_sample = dataset_m3[
    dataset_m3["quarter"] >= evaluation_start
].copy()

print("Evaluation observations:", len(evaluation_sample))

Evaluation observations: 37


## 1. Random Forest

Random Forest is used as a nonlinear machine-learning model that can capture interactions and nonlinear relationships between macroeconomic indicators without imposing a fixed functional form.

Hyperparameters are selected using time-series cross-validation on the initial pre-evaluation training sample and then held fixed throughout the expanding-window out-of-sample evaluation. The model itself is re-estimated at each forecast quarter using only information available before that quarter.

In [5]:
feature_cols_m1 = [
    col for col in dataset_m1.columns
    if col not in ["quarter", "gdp_qoq"]
]

feature_cols_m2 = [
    col for col in dataset_m2.columns
    if col not in ["quarter", "gdp_qoq"]
]

feature_cols_m3 = [
    col for col in dataset_m3.columns
    if col not in ["quarter", "gdp_qoq"]
]

print("M1 features:", len(feature_cols_m1))
print("M2 features:", len(feature_cols_m2))
print("M3 features:", len(feature_cols_m3))

M1 features: 2
M2 features: 6
M3 features: 18


In [6]:
def tune_random_forest(dataset, feature_cols):

    tuning_data = dataset[
        dataset["quarter"] < evaluation_start
    ].copy()

    X_tune = tuning_data[feature_cols]
    y_tune = tuning_data["gdp_qoq"]

    tscv = TimeSeriesSplit(n_splits=5)

    rf_base = RandomForestRegressor(
        n_estimators=300,
        random_state=42
    )

    param_grid = {
        "max_depth": [2, 3, 4, 5, None],
        "min_samples_leaf": [1, 2, 3, 5],
        "max_features": ["sqrt", 0.5, 1.0]
    }

    grid = GridSearchCV(
        estimator=rf_base,
        param_grid=param_grid,
        cv=tscv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    grid.fit(X_tune, y_tune)

    return grid.best_params_, -grid.best_score_

In [7]:
best_m1, cv_rmse_m1 = tune_random_forest(
    dataset_m1,
    feature_cols_m1
)

best_m2, cv_rmse_m2 = tune_random_forest(
    dataset_m2,
    feature_cols_m2
)

best_m3, cv_rmse_m3 = tune_random_forest(
    dataset_m3,
    feature_cols_m3
)

print("M1:", best_m1, "| CV RMSE:", cv_rmse_m1)
print("M2:", best_m2, "| CV RMSE:", cv_rmse_m2)
print("M3:", best_m3, "| CV RMSE:", cv_rmse_m3)

M1: {'max_depth': 2, 'max_features': 1.0, 'min_samples_leaf': 1} | CV RMSE: 0.637631992169936
M2: {'max_depth': 2, 'max_features': 0.5, 'min_samples_leaf': 1} | CV RMSE: 0.6974134071496491
M3: {'max_depth': 2, 'max_features': 0.5, 'min_samples_leaf': 1} | CV RMSE: 0.6907329278554372


In [8]:
rf_m1_results = []

for i in range (len(evaluation_sample)):
    forecast_quarter = evaluation_sample.iloc[i]["quarter"]

    rf_train = dataset_m1[
        dataset_m1["quarter"] < forecast_quarter].copy()

    rf_test = dataset_m1[
        dataset_m1["quarter"] == forecast_quarter].copy()

    X_train = rf_train [feature_cols_m1]
    y_train = rf_train["gdp_qoq"]

    X_test = rf_test[feature_cols_m1]

    rf_model = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        **best_m1      
    )
    
    rf_model.fit(X_train, y_train)

    rf_forecast = rf_model.predict(X_test)[0]

    actual = rf_test["gdp_qoq"].iloc[0]

    rf_m1_results.append({
        "quarter": forecast_quarter,
        "model": "Random Forest",
        "info_set": "M1",
        "actual": actual,
        "prediction": rf_forecast
    })
    



In [9]:
rf_m1_results = pd.DataFrame(rf_m1_results)

print("Number of forecasts:", len(rf_m1_results))

display(rf_m1_results.head())
display(rf_m1_results.tail())

Number of forecasts: 37


,quarter,model,info_set,actual,prediction
0,2017Q1,Random Forest,M1,1.0,0.598851
1,2017Q2,Random Forest,M1,0.9,0.621101
2,2017Q3,Random Forest,M1,0.7,0.618589
3,2017Q4,Random Forest,M1,0.8,0.638032
4,2018Q1,Random Forest,M1,0.1,0.647267


,quarter,model,info_set,actual,prediction
32,2025Q1,Random Forest,M1,0.6,0.373033
33,2025Q2,Random Forest,M1,0.2,0.325848
34,2025Q3,Random Forest,M1,0.1,0.375855
35,2025Q4,Random Forest,M1,0.1,0.376495
36,2026Q1,Random Forest,M1,0.6,0.370179


In [10]:
rf_m2_results = []

for i in range(len(evaluation_sample)):
    forecast_quarter = evaluation_sample.iloc[i]["quarter"]

    rf_train = dataset_m2[
        dataset_m2["quarter"] < forecast_quarter
    ].copy()

    rf_test = dataset_m2[
        dataset_m2["quarter"] == forecast_quarter
    ].copy()

    X_train = rf_train[feature_cols_m2]
    y_train = rf_train["gdp_qoq"]

    X_test = rf_test[feature_cols_m2]
    y_test = rf_test["gdp_qoq"]

    rf_model = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        **best_m2
    )

    rf_model.fit(X_train, y_train)

    prediction = rf_model.predict(X_test)[0]

    rf_m2_results.append({
        "quarter": forecast_quarter,
        "model": "Random Forest",
        "info_set": "M2",
        "actual": y_test.iloc[0],
        "prediction": prediction
    })

In [11]:
rf_m2_results = pd.DataFrame(rf_m2_results)

print("Number of forecasts:", len(rf_m2_results))

display(rf_m2_results.head())
display(rf_m2_results.tail())

Number of forecasts: 37


,quarter,model,info_set,actual,prediction
0,2017Q1,Random Forest,M2,1.0,0.570743
1,2017Q2,Random Forest,M2,0.9,0.573386
2,2017Q3,Random Forest,M2,0.7,0.617489
3,2017Q4,Random Forest,M2,0.8,0.577784
4,2018Q1,Random Forest,M2,0.1,0.592253


,quarter,model,info_set,actual,prediction
32,2025Q1,Random Forest,M2,0.6,0.326400
33,2025Q2,Random Forest,M2,0.2,0.328267
34,2025Q3,Random Forest,M2,0.1,0.349483
35,2025Q4,Random Forest,M2,0.1,0.336721
36,2026Q1,Random Forest,M2,0.6,0.324399


In [12]:
rf_m3_results = []

for i in range(len(evaluation_sample)):
    forecast_quarter = evaluation_sample.iloc[i]["quarter"]

    rf_train = dataset_m3[
        dataset_m3["quarter"] < forecast_quarter
    ].copy()

    rf_test = dataset_m3[
        dataset_m3["quarter"] == forecast_quarter
    ].copy()

    X_train = rf_train[feature_cols_m3]
    y_train = rf_train["gdp_qoq"]

    X_test = rf_test[feature_cols_m3]
    y_test = rf_test["gdp_qoq"]

    rf_model = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        **best_m3
    )

    rf_model.fit(X_train, y_train)

    prediction = rf_model.predict(X_test)[0]

    rf_m3_results.append({
        "quarter": forecast_quarter,
        "model": "Random Forest",
        "info_set": "M3",
        "actual": y_test.iloc[0],
        "prediction": prediction
    })

In [13]:
rf_m3_results = pd.DataFrame(rf_m3_results)

print("Number of forecasts:", len(rf_m3_results))

display(rf_m3_results.head())
display(rf_m3_results.tail())

Number of forecasts: 37


,quarter,model,info_set,actual,prediction
0,2017Q1,Random Forest,M3,1.0,0.528640
1,2017Q2,Random Forest,M3,0.9,0.537259
2,2017Q3,Random Forest,M3,0.7,0.591145
3,2017Q4,Random Forest,M3,0.8,0.575508
4,2018Q1,Random Forest,M3,0.1,0.553562


,quarter,model,info_set,actual,prediction
32,2025Q1,Random Forest,M3,0.6,0.389207
33,2025Q2,Random Forest,M3,0.2,0.356092
34,2025Q3,Random Forest,M3,0.1,0.383125
35,2025Q4,Random Forest,M3,0.1,0.324345
36,2026Q1,Random Forest,M3,0.6,0.381852


## 2. Gradient Boosting

Gradient Boosting is used as a second nonlinear machine-learning model.

Hyperparameters are selected using time-series cross-validation. The model is evaluated using the same expanding-window framework as the other models.

In [14]:
def tune_gradient_boosting(dataset, feature_cols):

    tuning_data = dataset[
        dataset["quarter"] < evaluation_start
    ].copy()

    X_tune = tuning_data[feature_cols]
    y_tune = tuning_data["gdp_qoq"]

    tscv = TimeSeriesSplit(n_splits=5)

    gb_base = GradientBoostingRegressor(
        random_state=42
    )

    param_grid = {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.01, 0.05, 0.1],
        "max_depth": [1, 2, 3],
        "min_samples_leaf": [1, 2, 3]
    }

    grid = GridSearchCV(
        estimator = gb_base,
        param_grid=param_grid,
        cv=tscv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    grid.fit(X_tune, y_tune)

    return grid.best_params_, -grid.best_score_

In [15]:
best_gb_m1, cv_gb_m1 = tune_gradient_boosting(
    dataset_m1,
    feature_cols_m1
)

best_gb_m2, cv_gb_m2 = tune_gradient_boosting(
    dataset_m2,
    feature_cols_m2
)

best_gb_m3, cv_gb_m3 = tune_gradient_boosting(
    dataset_m3,
    feature_cols_m3
)

print("M1:", best_gb_m1, "| CV RMSE:", round(cv_gb_m1, 3))
print("M2:", best_gb_m2, "| CV RMSE:", round(cv_gb_m2, 3))
print("M3:", best_gb_m3, "| CV RMSE:", round(cv_gb_m3, 3))

M1: {'learning_rate': 0.1, 'max_depth': 1, 'min_samples_leaf': 1, 'n_estimators': 200} | CV RMSE: 0.547
M2: {'learning_rate': 0.01, 'max_depth': 3, 'min_samples_leaf': 1, 'n_estimators': 100} | CV RMSE: 0.637
M3: {'learning_rate': 0.01, 'max_depth': 1, 'min_samples_leaf': 3, 'n_estimators': 200} | CV RMSE: 0.617


In [16]:
gb_m1_results = []

for i in range(len(evaluation_sample)):
    forecast_quarter = evaluation_sample.iloc[i]["quarter"]

    gb_train = dataset_m1[
        dataset_m1["quarter"] < forecast_quarter
    ].copy()

    gb_test = dataset_m1[
        dataset_m1["quarter"] == forecast_quarter
    ].copy()

    X_train = gb_train[feature_cols_m1]
    y_train = gb_train["gdp_qoq"]

    X_test = gb_test[feature_cols_m1]
    y_test = gb_test["gdp_qoq"]

    gb_model = GradientBoostingRegressor(
        random_state=42,
        **best_gb_m1
    )

    gb_model.fit(X_train, y_train)

    prediction = gb_model.predict(X_test)[0]

    gb_m1_results.append({
        "quarter": forecast_quarter,
        "model": "Gradient Boosting",
        "info_set": "M1",
        "actual": y_test.iloc[0],
        "prediction": prediction
    })

In [17]:
gb_m1_results = pd.DataFrame(gb_m1_results)

print("Number of forecasts:", len(gb_m1_results))

display(gb_m1_results.head())
display(gb_m1_results.tail())

Number of forecasts: 37


,quarter,model,info_set,actual,prediction
0,2017Q1,Gradient Boosting,M1,1.0,0.698537
1,2017Q2,Gradient Boosting,M1,0.9,0.741372
2,2017Q3,Gradient Boosting,M1,0.7,0.671567
3,2017Q4,Gradient Boosting,M1,0.8,0.734975
4,2018Q1,Gradient Boosting,M1,0.1,0.756525


,quarter,model,info_set,actual,prediction
32,2025Q1,Gradient Boosting,M1,0.6,0.656642
33,2025Q2,Gradient Boosting,M1,0.2,0.654804
34,2025Q3,Gradient Boosting,M1,0.1,0.647184
35,2025Q4,Gradient Boosting,M1,0.1,0.638091
36,2026Q1,Gradient Boosting,M1,0.6,0.628623


In [18]:
gb_m2_results = []

for i in range(len(evaluation_sample)):
    forecast_quarter = evaluation_sample.iloc[i]["quarter"]

    gb_train = dataset_m2[
        dataset_m2["quarter"] < forecast_quarter
    ].copy()

    gb_test = dataset_m2[
        dataset_m2["quarter"] == forecast_quarter
    ].copy()

    X_train = gb_train[feature_cols_m2]
    y_train = gb_train["gdp_qoq"]

    X_test = gb_test[feature_cols_m2]
    y_test = gb_test["gdp_qoq"]

    gb_model = GradientBoostingRegressor(
        random_state=42,
        **best_gb_m2
    )

    gb_model.fit(X_train, y_train)

    prediction = gb_model.predict(X_test)[0]

    gb_m2_results.append({
        "quarter": forecast_quarter,
        "model": "Gradient Boosting",
        "info_set": "M2",
        "actual": y_test.iloc[0],
        "prediction": prediction
    })

In [19]:
gb_m2_results = pd.DataFrame(gb_m2_results)

print("Number of forecasts:", len(gb_m2_results))

display(gb_m2_results.head())
display(gb_m2_results.tail())

Number of forecasts: 37


,quarter,model,info_set,actual,prediction
0,2017Q1,Gradient Boosting,M2,1.0,0.463209
1,2017Q2,Gradient Boosting,M2,0.9,0.675216
2,2017Q3,Gradient Boosting,M2,0.7,0.681542
3,2017Q4,Gradient Boosting,M2,0.8,0.683817
4,2018Q1,Gradient Boosting,M2,0.1,0.678631


,quarter,model,info_set,actual,prediction
32,2025Q1,Gradient Boosting,M2,0.6,0.354148
33,2025Q2,Gradient Boosting,M2,0.2,0.206953
34,2025Q3,Gradient Boosting,M2,0.1,0.369961
35,2025Q4,Gradient Boosting,M2,0.1,0.356921
36,2026Q1,Gradient Boosting,M2,0.6,0.347198


In [20]:
gb_m3_results = []

for i in range(len(evaluation_sample)):
    forecast_quarter = evaluation_sample.iloc[i]["quarter"]

    gb_train = dataset_m3[
        dataset_m3["quarter"] < forecast_quarter
    ].copy()

    gb_test = dataset_m3[
        dataset_m3["quarter"] == forecast_quarter
    ].copy()

    X_train = gb_train[feature_cols_m3]
    y_train = gb_train["gdp_qoq"]

    X_test = gb_test[feature_cols_m3]
    y_test = gb_test["gdp_qoq"]

    gb_model = GradientBoostingRegressor(
        random_state=42,
        **best_gb_m3
    )

    gb_model.fit(X_train, y_train)

    prediction = gb_model.predict(X_test)[0]

    gb_m3_results.append({
        "quarter": forecast_quarter,
        "model": "Gradient Boosting",
        "info_set": "M3",
        "actual": y_test.iloc[0],
        "prediction": prediction
    })

In [21]:
gb_m3_results = pd.DataFrame(gb_m3_results)

print("Number of forecasts:", len(gb_m3_results))

display(gb_m3_results.head())
display(gb_m3_results.tail())

Number of forecasts: 37


,quarter,model,info_set,actual,prediction
0,2017Q1,Gradient Boosting,M3,1.0,0.450088
1,2017Q2,Gradient Boosting,M3,0.9,0.482803
2,2017Q3,Gradient Boosting,M3,0.7,0.506869
3,2017Q4,Gradient Boosting,M3,0.8,0.516629
4,2018Q1,Gradient Boosting,M3,0.1,0.531816


,quarter,model,info_set,actual,prediction
32,2025Q1,Gradient Boosting,M3,0.6,0.446310
33,2025Q2,Gradient Boosting,M3,0.2,0.448443
34,2025Q3,Gradient Boosting,M3,0.1,0.445043
35,2025Q4,Gradient Boosting,M3,0.1,0.437265
36,2026Q1,Gradient Boosting,M3,0.6,0.432774


## 3. Save Forecast Results

Forecasts from the machine-learning models are saved for use in the final model evaluation.

In [22]:
rf_m1_results.to_csv(
    "../data/processed/rf_m1_forecasts.csv",
    index=False
)

rf_m2_results.to_csv(
    "../data/processed/rf_m2_forecasts.csv",
    index=False
)

rf_m3_results.to_csv(
    "../data/processed/rf_m3_forecasts.csv",
    index=False
)

gb_m1_results.to_csv(
    "../data/processed/gb_m1_forecasts.csv",
    index=False
)

gb_m2_results.to_csv(
    "../data/processed/gb_m2_forecasts.csv",
    index=False
)

gb_m3_results.to_csv(
    "../data/processed/gb_m3_forecasts.csv",
    index=False
)

print("Machine-learning forecasts saved.")

Machine-learning forecasts saved.
